In [ ]:
#pip install 'pymilvus[milvus-lite]==2.5.1'

In [ ]:
import os

from query_caller import QueryCaller

os.environ["LLM_MODEL"] = "granite4:latest"
os.environ["CHAT_ID"] = "3333"
caller = QueryCaller()

while True:
    user_input = input("> ")
    print(">", user_input)
    r1 = caller.supervisor_agent.send_query(user_input)

    print(">", r1)


/Users/fatihpolatli/Development/langchain-example/multi-agent/.venv/lib/python3.10/site-packages/pymilvus/client/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Supervisor Agent initialized
RAG Agent initialized
Custom Agent initialized


2026-04-25 20:56:00,572 [DEBUG][_create_connection]: Created new connection using: b84408ef05b84b13b2cbca02011bafb3 (async_milvus_client.py:545)


> how tall is a llama?
Sending query how tall is a llama?
================================== Ai Message ==================================
Tool Calls:
  rag_search (97e385f6-88d1-4cee-a268-b95cbc4b7727)
 Call ID: 97e385f6-88d1-4cee-a268-b95cbc4b7727
  Args:
    query: how tall is a llama
================================= Tool Message =================================
Name: rag_search

{'messages': [HumanMessage(content='how tall is a llama', additional_kwargs={}, response_metadata={}, id='79fe99a7-bc6e-4c22-bbd7-388c15a71947'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'granite4:latest', 'created_at': '2026-04-25T17:56:17.471491Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1139129584, 'load_duration': 43084417, 'prompt_eval_count': 231, 'prompt_eval_duration': 384629584, 'eval_count': 39, 'eval_duration': 700710955, 'logprobs': None, 'model_name': 'granite4:latest', 'model_provider': 'ollama'}, id='lc_run--019dc5c8-fb4b-7671-930d-c6a36ff66306-

In [ ]:

from vector_store_manager import VectorStoreManager


vc = VectorStoreManager()
vc.add_documents_to_vector_store("../llama.pdf")

In [ ]:
pip install -r requirements.txt

In [ ]:

%%bash
export LANGSMITH_TRACING="true"
export LANGSMITH_API_KEY=""
export LANGSMITH_PROJECT="test"
export LANGSMITH_ENDPOINT=https://eu.api.smith.langchain.com

source ./.venv/bin/activate

In [ ]:
from pymilvus import connections
URI = "./milvus_demo.db"
connections.connect(alias="default", uri=URI)

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import re

def clean_text(text):
    """Cleans and normalizes text."""
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces/newlines with a single space
    text = re.sub(r'[^\w\s.,!?]', '', text)  # Remove special characters except common punctuation
    return text.strip()

def load_and_process_pdf_documents(file_path):
    # Implement your PDF loading and processing logic here
    # This function should return a list of processed documents
    pdf_loader = PyPDFLoader(file_path)

    pdf_docs = pdf_loader.load()

    for doc in pdf_docs:
        doc.metadata["source"] = file_path
        doc.page_content = clean_text(doc.page_content)

    return pdf_docs


In [ ]:
from langchain_milvus import Milvus
from langchain_core.documents import Document
URI = "./milvus_demo.db"

'''

vector_store = Milvus.from_documents(load_and_process_pdf_documents("llama.pdf"), embeddings,collection_name="test", connection_args={"uri": URI},index_params={"index_type": "FLAT", "metric_type": "L2"},)
'''
vector_store = Milvus(collection_name="test", embedding_function=embeddings, connection_args={"uri": URI},
                        index_params={"index_type": "IVF_FLAT", "metric_type": "L2"})





In [ ]:
vector_store.add_documents(documents=load_and_process_pdf_documents("llama.pdf"))

In [ ]:
results = vector_store.similarity_search("What is llama?", k=3)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver


model = init_chat_model("granite4:latest", model_provider="ollama")

In [ ]:
from langchain.tools import tool

@tool
def create_calendar_event(  title: str,
    start_time: str,       # ISO format: "2024-01-15T14:00:00"
    end_time: str,         # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = "") -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
     # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

@tool
def send_email(
    to: list[str],  # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}"


@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]

@tool
def similarity_search(query: str, k: int = 3) -> list[Document]:
    """Search for similar documents in the vector store."""
    return vector_store.similarity_search(query, k=k)


In [ ]:
from langchain.agents import create_agent

CALENDAR_AGENT_PROMPT = (
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "If there is no suitable time slot, stop and confirm unavailability in your response. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

calendar_agent =  create_agent(model,  tools=[create_calendar_event, get_available_time_slots], system_prompt=CALENDAR_AGENT_PROMPT,
                           middleware=[HumanInTheLoopMiddleware(interrupt_on={"create_calendar_event":True},
                                                                description_prefix="Calendar event creation requires human approval. ")])

In [ ]:
query = "Schedule a team meeting next Tuesday at 2pm for 1 hour"

for step in calendar_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

In [ ]:


EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(model, tools=[send_email], system_prompt=EMAIL_AGENT_PROMPT,
                          middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email": True},
            description_prefix="Outbound email pending approval",
        ),
    ],)


In [ ]:
search_agent = create_agent(model,  system_prompt="You are a research assistant. Use search_and_answer to find information and answer questions.")

In [ ]:
rag_agent = create_agent(model, tools=[similarity_search], system_prompt="You are a research assistant. Use similarity_search to find relevant information from the document store to answer user queries.")

In [ ]:
query = "Send the design team a reminder about reviewing the new mockups"

for step in email_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

In [ ]:
from langchain_core.documents import Document

@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text

@tool
def search_web(query: str) -> str:
    """Perform a web search to gather information.

    Use this when the user asks for information that may not be in the agent's
    knowledge base or when they explicitly request a search. This can help
    provide up-to-date information or context for other tasks.

    Input: Natural language search query (e.g., 'What is the latest on the
    project deadline?')
    """

    result = search_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })

    # Stub: In practice, this would call a search API like Bing Search API, Google Custom Search, etc.
    return f"Search results for: {query} is: {result['messages'][-1].text}"

@tool
def rag_search(query: str) -> list[Document]:
    """Perform a RAG search over the document store.

    Use this when the user asks questions that can be answered by the documents
    in the vector store. This allows the agent to provide informed answers based
    on the content of the documents.

    Input: Natural language question (e.g., 'What are the key points from the
    latest project report?')
    """

    results = rag_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    

    return results;

In [ ]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
    "You can make search on the web to find information when needed. "
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email, rag_search],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=InMemorySaver()
)

In [ ]:
#query = "Schedule a team standup for tomorrow at 9am"
query = "How does a llama look like? show a picture if possible"

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    config={"configurable": {"thread_id": "1"}}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

In [ ]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

In [ ]:
from langgraph.types import Command 
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

config = {"configurable": {"thread_id": "6"}}

resume = {}
for interrupt_ in interrupts:
    if interrupt_.id == "e21cf769d62cfba377dd419c002f0a76":
        # Edit email
        edited_action = interrupt_.value["action_requests"][0].copy()
        edited_action["args"]["subject"] = "Mockups reminder"
        resume[interrupt_.id] = {
            "decisions": [{"type": "edit", "edited_action": edited_action}]
        }
    else:
        resume[interrupt_.id] = {"decisions": [{"type": "approve"}]}

interrupts = []
for step in supervisor_agent.stream(
     Command(resume=resume),
    config,
):
    for update in step.values():
        if isinstance(update, dict):
            for message in update.get("messages", []):
                message.pretty_print()
        else:
            interrupt_ = update[0]
            interrupts.append(interrupt_)
            print(f"\nINTERRUPTED: {interrupt_.id}")